Lets load both datasets for both fiscal years:

In [23]:

import pandas as pd

year1 = pd.read_csv('data/online_retail_I.csv')
year2 = pd.read_csv('data/online_retail_II.csv')

display(year1.head())
year2.head()






,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/09 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/09 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/09 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/09 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/09 7:45,1.25,13085.0,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


now lets combine them into 1 dataframe. because I want to assign a new index to each new record as we are joining 2 csv files, I use ignore_index=True

In [27]:
df = pd.concat([year1, year2], ignore_index=True)

print("Combined dataset:", df.shape)

display(df.head())
df.tail()


Combined dataset: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,12/1/09 7:45,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,12/1/09 7:45,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,12/1/09 7:45,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,12/1/09 7:45,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,12/1/09 7:45,1.25,13085.0,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/11 12:50,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/11 12:50,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/11 12:50,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12/9/11 12:50,4.95,12680.0,France
1067370,581587,POST,POSTAGE,1,12/9/11 12:50,18.00,12680.0,France


Now, before the data cleaning, lets analyze the df as a whole with the following methods:

In [41]:
df.info()
df.describe(include='all')
df.isnull().sum()
print("Duplicate rows:", df.duplicated().sum())

print("Countries:", df['Country'].nunique())
print("Distinct products:", df['StockCode'].nunique())
print("Distinct invoices:", df['Invoice'].nunique())
print("Distinct customers:", df['Customer ID'].nunique())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
Duplicate rows: 34335
Countries: 43
Distinct products: 5305
Distinct invoices: 53628
Distinct customers: 5942


The results are that the df contains over 1 million records. Other interesting observations include that there are 43 countries 5305 different products and 5942 different customers. Surprisingly, aproximately 21% of Customer IDs are null and also description is missing for some transactions, but overall, all other features do not have null values. Also, there is no unique id for each record and I've found that there are 34335 duplicate rows.

Next, since it is an e-commerce dataset and customers may have been returning produnts, negative quantities, negative prices, and I also noticed that some invoices started with C,  may represent returned products or canceled transactions. So I will also check for that:

In [ ]:

print("Negative quantities:", (df['Quantity'] < 0).sum())
print("Zero quantities:", (df['Quantity'] == 0).sum())

print("Negative prices:", (df['Price'] < 0).sum())
print("Zero prices:", (df['Price'] == 0).sum())

print("Cancelled invoices:", df['Invoice'].str.startswith('C').sum())



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,12/1/09 11:34,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,12/1/09 11:34,3.75,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,12/1/09 11:34,3.75,16329.0,United Kingdom
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,12/1/09 11:34,0.65,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,12/1/09 11:34,0.85,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,12/1/09 11:34,3.75,16329.0,United Kingdom
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,12/1/09 11:34,1.95,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,12/1/09 11:34,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,12/1/09 11:34,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,12/1/09 11:34,3.75,16329.0,United Kingdom


Negative quantities: 22950
Zero quantities: 0
Negative prices: 5
Zero prices: 6202
Cancelled invoices: 19494


/var/folders/h4/g727ynkx15770svbrv8vbnn00000gn/T/ipykernel_89200/3117823113.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2011-12-09 12:50:00


Now, after I gathered this information, I will start the data cleaning! I will make a few changes to the dataset:

-Remove exact duplicate rows to avoid counting the same transaction more than once.
-Exclude invoices that start with "C", since they represent cancelled transactions.
-Keep records with negative quantities because they may represent product returns. These will be considered when analyzing revenue.
-Keep records with missing product descriptions and Customer IDs. These records can still be used for general sales analysis, but some product-level and customer-level analyses may not include them.

Since cancelled invoices are excluded, the analysis will focus on sales transactions and will not fully reflect the impact of cancellations on revenue, so that is one of the limitations.

In [ ]:
df_cleaned = df.copy()

# remove cancelled invoices
df_cleaned = df_cleaned[~df_cleaned['Invoice'].str.startswith('C')]

# remove exact duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# remove records with zero or negative prices
df_cleaned = df_cleaned[df_cleaned['Price'] > 0]

df_cleaned.head(100)
df_cleaned.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1007913 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007913 non-null  object        
 1   StockCode    1007913 non-null  object        
 2   Description  1007913 non-null  object        
 3   Quantity     1007913 non-null  int64         
 4   InvoiceDate  1007913 non-null  datetime64[ns]
 5   Price        1007913 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007913 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 69.2+ MB


After the light cleaning, ~59k rows were removed. Now, I want to add a revenue column to calculate revenue to the df. To calculate total revenue, I multiply the quantity of each product by its unit price. I then sum the revenue across all transactions to get the total net revenue. Negative quantities are included in this calculation, so product returns reduce the total revenue.

In [61]:
df_cleaned['Revenue'] = df_cleaned['Quantity'] * df_cleaned['Price']
df_cleaned.head(15)
revenue = round(df_cleaned['Revenue'].sum(),2)
print(f'${revenue}')

$20476260.45
